# Lesson 26 — Differentiable Optimization

## Learning objectives

1. Differentiate an *argmin* with respect to its parameters via the implicit-function theorem.
2. Use an optimization layer — `discopt`'s KKT sensitivities, OptNet, or cvxpylayers — inside a differentiable pipeline {cite:p}`Amos2017,Agrawal2019`.
3. Recognize the contexts where differentiable optimization wins.
4. Understand the link to decision-focused learning (Lesson 27).

## 1. The implicit-function theorem

For an optimization problem $x^\star(\theta) = \arg\min_x f(x, \theta)$, suppose $\nabla_x f(x^\star, \theta) = 0$. Differentiating implicitly:

$$\nabla_{xx}^2 f \cdot \frac{\partial x^\star}{\partial \theta} + \nabla_{x \theta}^2 f = 0,$$

so

$$\frac{\partial x^\star}{\partial \theta} = -[\nabla_{xx}^2 f]^{-1} \nabla_{x \theta}^2 f.$$

Same idea works for KKT systems with active inequality constraints — differentiate the KKT system implicitly {cite:p}`Amos2017`.

## 2. OptNet

Embed a QP layer

$$z(\theta) = \arg\min_z \tfrac{1}{2} z^\top Q(\theta) z + p(\theta)^\top z \;\;\text{s.t.}\;\; A(\theta) z = b(\theta), \; G(\theta) z \le h(\theta)$$

inside a neural network. The forward pass solves the QP; the backward pass differentiates through the KKT system. {cite:p}`Amos2017`.

For convex programs in DCP form, `cvxpylayers` {cite:p}`Agrawal2019` automates this — define your CP, get a PyTorch / JAX layer.

## 3. When differentiable optimization wins

- **Constraint as inductive bias:** the network output is *guaranteed* to satisfy a physical constraint (e.g., $\sum_i w_i = 1$, a probability).
- **Decision-focused learning:** train upstream prediction to minimize downstream decision regret (Lesson 27).
- **Differentiable physics:** PDEs as optimization layers.

When *not* to use it: when the constraint is a soft preference (just penalize), when the QP is large (forward + backward both expensive), or when discrete constraints dominate (differentiating MILP is hard).

## 4. Differentiating MIP

Strict integer constraints are non-differentiable; approximations:

- **LP relaxation + STE:** use LP-relaxed gradient on the backward pass.
- **Smoothed perturbed argmax:** add Gumbel noise, take softmax {cite:p}`Mandi2020`.
- **Sub-gradient via Fenchel-Young loss** {cite:p}`Elmachtoub2022`.

These give *useful* but biased gradients.

### Differentiable optimization in `discopt`

`discopt` differentiates a solved model through the implicit-function
theorem applied to the KKT system at the returned primal-dual point —
the sIPOPT-style sensitivity of {cite:p}`Pirnay2012`, the same idea as
OptNet {cite:p}`Amos2017`. Declare the inputs you want derivatives with
respect to as scalar `Parameter`s, solve, and ask:

| Call | Returns |
|---|---|
| `r.gradient(p)` (or `solve(sensitivity=True)`) | $d\,\text{obj}^\star/dp$ by the envelope theorem |
| `s = m.sensitivity(wrt=[p1, p2, ...])` | `s.dx_dp`, `s.d(var)` ($dx^\star/dp$ in the variable's shape), `s.d(expr)` (total derivative of any expression through $x^\star(p)$), `s.dobj_dp`, `s.predict({p: v})` (first-order re-solve) |
| `s.as_layer()` | a JAX callable $p \mapsto x^\star$ for `jax.grad` / `jax.jacobian` / `jit` / `vmap` |
| `dm.argmin(...)` | an inner NLP embedded as a differentiable block of an *outer* model |

The contract matters: continuous variables only, derivatives belong to the
active set at the returned (for nonconvex models: *local*) KKT point, and
at an active-set change $x^\star(p)$ has a kink. Integer variables are
refused here; the fix-and-differentiate gradient for MILP/MIQP (fix the
integers at the optimum, differentiate the restriction — §4's "useful but
biased" family) is in `discopt._relax.differentiable.differentiable_solve`,
an internal module shown in discopt's `differentiable_milp` notebook.

For exercises 2 and 3 you may use `discopt` (cell below) or the external
tools **`cvxpylayers`** (PyTorch / JAX) and **`qpth`** (the OptNet QP
solver). A typical cvxpylayers workflow:

```python
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

mu_p = cp.Parameter(n_assets)
w_v  = cp.Variable(n_assets)
prob = cp.Problem(
    cp.Minimize(0.5 * cp.quad_form(w_v, Sigma) - mu_p @ w_v),
    [cp.sum(w_v) == 1, w_v >= 0],
)
layer = CvxpyLayer(prob, parameters=[mu_p], variables=[w_v])
# layer(torch_mu) is now differentiable w.r.t. torch_mu.
```

The decision-focused exercise below (E3) gives you the loss-function pattern;
the optimization layer is what makes the gradient meaningful.

In [ ]:
import numpy as np
import discopt as do
import discopt.modeling as dm
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# The same portfolio QP as the cvxpylayers snippet, differentiated by discopt.
Sigma = np.array([[0.10, 0.02, 0.01],
                  [0.02, 0.08, 0.00],
                  [0.01, 0.00, 0.12]])
mu0 = np.array([0.10, 0.12, 0.08])
n_assets = len(mu0)

m = dm.Model("portfolio")
mu = [m.parameter(f"mu{i}", value=mu0[i]) for i in range(n_assets)]   # scalar Parameters
w = m.continuous("w", shape=(n_assets,), lb=0, ub=1)
m.subject_to(dm.sum(w) == 1)
m.minimize(0.5 * (w @ (Sigma @ w)) - sum(mu[i] * w[i] for i in range(n_assets)))
r = m.solve()
print("status:", r.status, " w* =", r.value(w).round(4))

s = m.sensitivity(wrt=mu)
print("dw*/dmu (rows: w_i, cols: mu_j):\n", np.asarray(s.d(w)).round(3))
print("rows sum to zero because sum(w) = 1 is active:", np.asarray(s.d(w)).sum(axis=0).round(8))

# As a JAX layer: compose with jax.grad / jax.jacobian like any other function.
layer = s.as_layer()
print("jax.jacobian agrees:", np.allclose(jax.jacobian(layer)(jnp.array(mu0)), s.d(w)))


## 5. Caveats

- Forward pass: solver call per example per batch.
- Backward pass: extra solve / KKT linear system.
- Numerical conditioning: differentiating near the boundary of the feasible region or near degeneracy is unstable.

## References
{cite:p}`Amos2017,Agrawal2019,Mandi2020,Elmachtoub2022`.